In [1]:
# Install pmdarima package
!pip install pmdarima

In [2]:
# Import necessary libraries
import pmdarima as pm
import numpy as np
import pandas as pd
import requests
import jdatetime
import calendar
import re
import urllib3
urllib3.disable_warnings()
from persiantools import characters

In [3]:
# Get Stocks data function
headers = {'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_10_1) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/39.0.2171.95 Safari/537.36'}
################################################################################################################################################################################
################################################################################################################################################################################
def __Check_JDate_Validity__(date, key_word):
    try:
        if(len(date.split('-')[0])==4):
            date = jdatetime.date(year=int(date.split('-')[0]), month=int(date.split('-')[1]), day=int(date.split('-')[2]))
            date = f'{date.year:04}-{date.month:02}-{date.day:02}'
            return date
        else:
            print(f'Please enter valid {key_word} date in YYYY-MM-DD format')
    except:
        if(len(date)==10):
            print(f'Please enter valid {key_word} date')
            return
        else:
            print(f'Please enter valid {key_word} date in YYYY-MM-DD format')
            
################################################################################################################################################################################
################################################################################################################################################################################
def get_tse_webid(stock:str = 'پترول') -> pd.DataFrame:
    """
    Takes ticker or firm's full name, does a live search in TSE new website and returns a multi-index Pandas dataframe that contains the following columns:
    
    Ticker: Symbol in the Tehran Stock Exchange.
    Active: 1 shows the market in which the stock is currently trading. 
    Name: firm's full name in the relevant market.
    WebID: A numeric code that can be used for building request links and crawling the financial data of the given stock.
    Market: Market name in Tehran Stock Exchange, markets the stock was traded in and is trading now (بورس، فرابورس، پایه زرد، پایه نارنجی، پایه قرمز).
    
    :param stock: (str) Ticker or firm's full name. 
    :return: (pd.DataFrame) A dataframe that contains Ticker, Active, WebID, Name and Market columns for the requested stock.
    """
    
    # basic search function: searches for and cleans the search results
    def srch_req(srch_key):
        srch_page = requests.get(f'http://cdn.tsetmc.com/api/Instrument/GetInstrumentSearch/{srch_key}', headers=headers)
        srch_res = pd.DataFrame(srch_page.json()['instrumentSearch'])
        srch_res = srch_res[['lVal18AFC','lVal30','insCode','lastDate','cgrValCot']]
        srch_res.columns = ['Ticker','Name','WebID','Active','Market']
        srch_res['Name'] = srch_res['Name'].apply(lambda x : characters.ar_to_fa(' '.join([i.strip() for i in x.split('\u200c')]).strip()))
        srch_res['Ticker'] = srch_res['Ticker'].apply(lambda x : characters.ar_to_fa(''.join(x.split('\u200c')).strip()))
        srch_res['NameSplit'] = srch_res['Name'].apply(lambda x : ''.join(x.split()).strip())
        srch_res['SymbolSplit'] = srch_res['Ticker'].apply(lambda x : ''.join(x.split()).strip())
        srch_res['Active'] = pd.to_numeric(srch_res['Active'])
        srch_res = srch_res.sort_values('Ticker')
        srch_res = pd.DataFrame(srch_res[['Name','WebID','NameSplit','SymbolSplit','Market']].values, columns=['Name','WebID',
                                'NameSplit','SymbolSplit','Market'], index=pd.MultiIndex.from_frame(srch_res[['Ticker','Active']]))
        return srch_res
    
    # checking function inputs
    if type(stock) != str:
        print('Please Enetr a Valid Ticker or Name!')
        return False
    
    # special case that can not be found using ticker: convert ticker to full name!
    if(stock=='آ س پ'):
        stock = 'آ.س.پ'
        
    # generating search keys
    stock = characters.ar_to_fa(''.join(stock.split('\u200c')).strip())
    first_name = stock.split()[0]
    stock = ''.join(stock.split())
    
    # start searching using keys, cleaning data, checking search results and handling special cases (ticker or full name)
    data = srch_req(first_name)
    df_symbol = data[data['SymbolSplit'] == stock]
    df_name = data[data['NameSplit'] == stock]
    
    # matching search results with search key, cleaning the data and adding market data  
    if len(df_symbol) > 0 :
        df_symbol = df_symbol.sort_index(level=1,ascending=False).drop(['NameSplit','SymbolSplit'], axis=1)
        df_symbol['Market'] = df_symbol['Market'].apply(lambda x: re.sub('[0-9]', '', x))
        df_symbol['Market'] = df_symbol['Market'].map({'N':'بورس', 'Z':'فرابورس', 'D':'فرابورس', 'A':'پایه زرد', 'P':'پایه زرد', 'C':'پایه نارنجی', 'L':'پایه قرمز',
                                                       'W':'کوچک و متوسط فرابورس', 'V':'کوچک و متوسط فرابورس',})
        df_symbol['Market'] = df_symbol['Market'].fillna('نامعلوم')
        return df_symbol
    elif len(df_name) > 0 :
        symbol = df_name.index[0][0]
        data = srch_req(symbol)
        symbol = characters.ar_to_fa(''.join(symbol.split('\u200c')).strip())
        df_symbol = data[data.index.get_level_values('Ticker') == symbol]
        if len(df_symbol) > 0 :
            df_symbol = df_symbol.sort_index(level=1, ascending=False).drop(['NameSplit','SymbolSplit'], axis=1)
            df_symbol['Market'] = df_symbol['Market'].apply(lambda x: re.sub('[0-9]', '', x))
            df_symbol['Market'] = df_symbol['Market'].map({'N':'بورس', 'Z':'فرابورس', 'D':'فرابورس', 'A':'پایه زرد', 'P':'پایه زرد', 'C':'پایه نارنجی', 'L':'پایه قرمز',
                                                           'W':'کوچک و متوسط فرابورس', 'V':'کوچک و متوسط فرابورس',})
            df_symbol['Market'] = df_symbol['Market'].fillna('نامعلوم')
            return df_symbol
    
    # invalid entry
    print('Please Enetr a Valid Ticker or Name!')
    
    return False


def __Get_TSE_WebID__(stock):
    # search TSE function ------------------------------------------------------------------------------------------------------------
    def request(name):
        page = requests.get(f'http://old.tsetmc.com/tsev2/data/search.aspx?skey={name}', headers=headers)
        data = []
        for i in page.text.split(';') :
            try :
                i = i.split(',')
                data.append([i[0],i[1],i[2],i[7],i[-1]])
            except :
                pass
        data = pd.DataFrame(data, columns=['Ticker','Name','WEB-ID','Active','Market'])
        data['Name'] = data['Name'].apply(lambda x : characters.ar_to_fa(' '.join([i.strip() for i in x.split('\u200c')]).strip()))
        data['Ticker'] = data['Ticker'].apply(lambda x : characters.ar_to_fa(''.join(x.split('\u200c')).strip()))
        data['Name-Split'] = data['Name'].apply(lambda x : ''.join(x.split()).strip())
        data['Symbol-Split'] = data['Ticker'].apply(lambda x : ''.join(x.split()).strip())
        data['Active'] = pd.to_numeric(data['Active'])
        data = data.sort_values('Ticker')
        data = pd.DataFrame(data[['Name','WEB-ID','Name-Split','Symbol-Split','Market']].values, columns=['Name','WEB-ID',
                            'Name-Split','Symbol-Split','Market'], index=pd.MultiIndex.from_frame(data[['Ticker','Active']]))
        return data
    #---------------------------------------------------------------------------------------------------------------------------------
    if type(stock) != str:
        print('Please Enetr a Valid Ticker or Name!')
        return False
    if(stock=='آ س پ'):
        stock = 'آ.س.پ'
    # cleaning input search key
    stock = characters.ar_to_fa(''.join(stock.split('\u200c')).strip())
    first_name = stock.split()[0]
    stock = ''.join(stock.split())
    # search TSE and process:
    data = request(first_name)
    df_symbol = data[data['Symbol-Split'] == stock]
    df_name = data[data['Name-Split'] == stock]
    if len(df_symbol) > 0 :
        df_symbol = df_symbol.sort_index(level=1,ascending=False).drop(['Name-Split','Symbol-Split'], axis=1)
        df_symbol['Market'] = df_symbol['Market'].apply(lambda x: re.sub('[0-9]', '', x))
        df_symbol['Market'] = df_symbol['Market'].map({'N':'بورس', 'Z':'فرابورس', 'D':'فرابورس', 'A':'پایه زرد', 'P':'پایه زرد', 'C':'پایه نارنجی', 'L':'پایه قرمز',
                                                       'W':'کوچک و متوسط فرابورس', 'V':'کوچک و متوسط فرابورس',})
        df_symbol['Market'] = df_symbol['Market'].fillna('نامعلوم')
        return df_symbol
    elif len(df_name) > 0 :
        symbol = df_name.index[0][0]
        data = request(symbol)
        symbol = characters.ar_to_fa(''.join(symbol.split('\u200c')).strip())
        df_symbol = data[data.index.get_level_values('Ticker') == symbol]
        if len(df_symbol) > 0 :
            df_symbol = df_symbol.sort_index(level=1,ascending=False).drop(['Name-Split','Symbol-Split'], axis=1)
            df_symbol['Market'] = df_symbol['Market'].apply(lambda x: re.sub('[0-9]', '', x))
            df_symbol['Market'] = df_symbol['Market'].map({'N':'بورس', 'Z':'فرابورس', 'D':'فرابورس', 'A':'پایه زرد', 'P':'پایه زرد', 'C':'پایه نارنجی', 'L':'پایه قرمز',
                                                           'W':'کوچک و متوسط فرابورس', 'V':'کوچک و متوسط فرابورس',})
            df_symbol['Market'] = df_symbol['Market'].fillna('نامعلوم')
            return df_symbol
    print('Please Enetr a Valid Ticker or Name!')
    return False

################################################################################################################################################################################
################################################################################################################################################################################

def get_price_history(stock:str = 'خودرو', 
                      start_date:str = '1400-01-01', 
                      end_date:str = '1401-01-01', 
                      ignore_date:bool = False, 
                      adjust_price:bool = False, 
                      show_weekday:bool = False, 
                      double_date:bool = False) -> pd.DataFrame:
    
    """
    Takes ticker or firm's full name and returns a Pandas dataframe that contains the following columns:
    
    J-Date: Jalali date, as index
    Date: Gregorian date
    Weekday: Name of weekdays
    Open: Opening price of day
    High: Maximum price of day 
    Low: Minimum price of day
    Close: Closing price of day (آخرین قیمت)
    Final: Weighted closing price of day (قیمت پایانی)
    Volume: Traded volume of day
    Value: Traded value of day in IRAN's Rial
    No: Number of trades in a day
    Ticker: Ticker/Symbol
    Name: Firm's full name
    Market: Market, the stock is traded in
    Adj Open: Adjusted opening price of day for stock-splits and dividends
    Adj High: Adjusted maximum price of day for stock-splits and dividends
    Adj Low: Adjusted minimum price of day for stock-splits and dividends
    Adj Close: Adjusted closing price of day for stock-splits and dividends
    Adj Final: Adjusted weighted closing price of day for stock-splits and dividends
    
     * All data are taken from the new website of Tehran Stock Exchange.

    :param stock: (str) Ticker or firm's full name. 
    :param start_date: (str) Jalali date for starting day of historical price data in YYYY-MM-DD format.
    :param end_date: (str) Jalali date for ending day of historical price data in YYYY-MM-DD format.
    :param ignore_date: (bool) Ignores start_date and end_date and returns all available price history, if set to True.
    :param adjust_price: (bool) Adjusts price for stock-splits and dividends, if set to True.
    :param show_weekday: (bool) Shows weekdays in the output, if set to True.
    :param double_date: (bool) Shows Gregorian date in the output, if set to True.
    :return: (pd.DataFrame) A dataframe that contains J-Date as index and Date, Weekday, Open, High, Low, Close, Final, Volume, Value, No, Ticker, Name, Market, Adj Open, Adj High, Adj Low, Adj Close, Adj Final.
    """

    # basic request and data cleaning function for historical price data of a ticker for a given market 
    def get_price_data(ticker_no, ticker, name, market):
        r = requests.get(f'http://cdn.tsetmc.com/api/ClosingPrice/GetClosingPriceDailyList/{ticker_no}/0', headers=headers)
        df_history = pd.DataFrame(r.json()['closingPriceDaily'])
        columns=['Date','High','Low','Final','Close','Open','Y-Final','Value','Volume','No']
        df_history = df_history[['dEven','priceMax','priceMin','pClosing','pDrCotVal','priceFirst','priceYesterday','qTotCap','qTotTran5J','zTotTran']]
        df_history.columns = ['Date','High','Low','Final','Close','Open','Y-Final','Value','Volume','No']
        df_history['Date'] = df_history['Date'].apply(lambda x: str(x))
        df_history['Date'] = df_history['Date'].apply(lambda x: f'{x[:4]}-{x[4:6]}-{x[-2:]}')
        df_history['Date']=pd.to_datetime(df_history['Date'])
        df_history = df_history[df_history['No']!=0]
        df_history['Ticker'] = ticker
        df_history['Name'] = name
        df_history['Market'] = market
        df_history = df_history.set_index('Date')
        return df_history
    
    # check to see if the entry start and end dates are valid or not
    if(not ignore_date):
        start_date = __Check_JDate_Validity__(start_date,key_word="'START'")
        if(start_date==None):
            return
        end_date = __Check_JDate_Validity__(end_date,key_word="'END'")
        if(end_date==None):
            return
        start = jdatetime.date(year=int(start_date.split('-')[0]), month=int(start_date.split('-')[1]), day=int(start_date.split('-')[2]))
        end = jdatetime.date(year=int(end_date.split('-')[0]), month=int(end_date.split('-')[1]), day=int(end_date.split('-')[2]))
        if(start>end):
            print('Start date must be a day before end date!')
            return
    
    # search for WebIDs
    ticker_no_df = get_tse_webid(stock)
    if(type(ticker_no_df)==bool):
        return
    
    # create an empty dataframe:
    df_history = pd.DataFrame({},columns=['Date','High','Low','Final','Close','Open','Y-Final','Value','Volume','No','Ticker','Name','Market']).set_index('Date')
    
    # loop to get data from different pages of a ticker:
    for index, row in (ticker_no_df.reset_index()).iterrows():
        try:
            df_temp = get_price_data(ticker_no = row['WebID'],ticker = row['Ticker'],name = row['Name'],market = row['Market'])
            df_history = pd.concat([df_history,df_temp])
        except:
            pass
        
    # sort based on dated index:
    df_history = df_history.sort_index(ascending=True)
    df_history = df_history.reset_index()
    
    # add weekdays and j-date columns:
    df_history['Weekday']=df_history['Date'].dt.weekday
    df_history['Weekday'] = df_history['Weekday'].apply(lambda x: calendar.day_name[x])
    df_history['J-Date']=df_history['Date'].apply(lambda x: str(jdatetime.date.fromgregorian(date=x.date())))
    df_history = df_history.set_index('J-Date')
    
    # rearrange columns and convert some columns to numeric
    df_history=df_history[['Date','Weekday','Y-Final','Open','High','Low','Close','Final','Volume','Value','No','Ticker','Name','Market']]
    cols = ['Y-Final','Open','High','Low','Close','Final','Volume','No','Value']
    df_history[cols] = df_history[cols].apply(pd.to_numeric, axis=1)
    

    # find stock moves between markets and adjust for nominal price in the new market, if necessary
    df_history['Final(+1)'] = df_history['Final'].shift(+1)          
    df_history['Market(+1)'] = df_history['Market'].shift(+1)        
    df_history['temp'] = df_history.apply(lambda x: x['Y-Final'] if((x['Y-Final']!=0)and(x['Y-Final']!=1000)) 
                                          else (x['Y-Final'] if((x['Market(+1)']==x['Market'])or(pd.isnull(x['Final(+1)']))) 
                                          else x['Final(+1)']),axis = 1)
    df_history['Y-Final'] = df_history['temp']
    df_history.drop(columns=['Final(+1)','temp','Market(+1)'],inplace=True)
    
    # convert numbers to int because we do not have less than Rial, just for clean outputs!
    for col in cols:
        df_history[col] = df_history[col].apply(lambda x: int(x))

    # Adjust price data, if requested:
    if(adjust_price):
        df_history['COEF'] = (df_history['Y-Final'].shift(-1)/df_history['Final']).fillna(1.0)
        df_history['ADJ-COEF']=df_history.iloc[::-1]['COEF'].cumprod().iloc[::-1]
        df_history['Adj Open'] = (df_history['Open']*df_history['ADJ-COEF']).apply(lambda x: int(x))
        df_history['Adj High'] = (df_history['High']*df_history['ADJ-COEF']).apply(lambda x: int(x))
        df_history['Adj Low'] = (df_history['Low']*df_history['ADJ-COEF']).apply(lambda x: int(x))
        df_history['Adj Close'] = (df_history['Close']*df_history['ADJ-COEF']).apply(lambda x: int(x))
        df_history['Adj Final'] = (df_history['Final']*df_history['ADJ-COEF']).apply(lambda x: int(x))
        df_history.drop(columns=['COEF','ADJ-COEF'],inplace=True)
    
    # drop weekdays if not requested
    if(not show_weekday):
        df_history.drop(columns=['Weekday'],inplace=True)
    
    # drop Gregorian date if not requested
    if(not double_date):
        df_history.drop(columns=['Date'],inplace=True)
    
    # drop yesterday's final price!
    df_history.drop(columns=['Y-Final'],inplace=True)
    
    # slice requested time window, if requested:
    if(not ignore_date):
        df_history = df_history[start_date:end_date]
        
    return df_history


def Get_Price_History(stock = 'خودرو', start_date = '1400-01-01', end_date='1401-01-01', ignore_date = False, adjust_price = False, show_weekday = False, double_date = False):
    """
    دریافت سابقه قیمت یک سهم در روزهای معاملاتی بین تاریخ شروع و پایان
    قابلیت دریافت همه سابقه قیمت بدون توجه به تاریخ شروع و پایان
    قابلیت تعدیل قیمت برای سود نقدی و افزایش سرمایه با احتساب آورده
    قابلیت ارائه تاریخ میلادی علاوه بر تاریخ شمسی، قابلیت نمایش روزهای هفته
    """
    # a function to get price data from a given page ----------------------------------------------------------------------------------
    def get_price_data(ticker_no,ticker,name, data_part):
        r = requests.get(f'http://old.tsetmc.com/tsev2/data/InstTradeHistory.aspx?i={ticker_no}&Top=999999&A=0', headers=headers)
        df_history=pd.DataFrame(r.text.split(';'))
        columns=['Date','High','Low','Final','Close','Open','Y-Final','Value','Volume','No']
        #split data into defined columns
        df_history[columns] = df_history[0].str.split("@",expand=True)
        # drop old column 0
        df_history.drop(columns=[0],inplace=True)
        df_history.dropna(inplace=True)
        df_history['Date']=pd.to_datetime(df_history['Date'])
        df_history['Ticker'] = ticker
        df_history['Name'] = name
        df_history['Market'] = data_part
        df_history = df_history.set_index('Date')
        return df_history
    # ----------------------------------------------------------------------------------------------------------------------------------
    # check date validity
    if(not ignore_date):
        start_date = __Check_JDate_Validity__(start_date,key_word="'START'")
        if(start_date==None):
            return
        end_date = __Check_JDate_Validity__(end_date,key_word="'END'")
        if(end_date==None):
            return
        start = jdatetime.date(year=int(start_date.split('-')[0]), month=int(start_date.split('-')[1]), day=int(start_date.split('-')[2]))
        end = jdatetime.date(year=int(end_date.split('-')[0]), month=int(end_date.split('-')[1]), day=int(end_date.split('-')[2]))
        if(start>end):
            print('Start date must be a day before end date!')
            return
    #---------------------------------------------------------------------------------------------------------------------------------------
    # find web-ids 
    ticker_no_df = __Get_TSE_WebID__(stock)
    if(type(ticker_no_df)==bool):
        return
    # create an empty dataframe:
    df_history = pd.DataFrame({},columns=['Date','High','Low','Final','Close','Open','Y-Final','Value','Volume','No','Ticker','Name','Market']).set_index('Date')
    # loop to get data from different pages of a ticker:
    for index, row in (ticker_no_df.reset_index()).iterrows():
        try:
            df_temp = get_price_data(ticker_no = row['WEB-ID'],ticker = row['Ticker'],name = row['Name'],data_part = row['Market'])
            df_history = pd.concat([df_history,df_temp])
        except:
            pass
    # sort index and reverse the order for more processes:
    df_history = df_history.sort_index(ascending=True)
    df_history = df_history.reset_index()
    # determining week days:
    df_history['Weekday']=df_history['Date'].dt.weekday
    df_history['Weekday'] = df_history['Weekday'].apply(lambda x: calendar.day_name[x])
    df_history['J-Date']=df_history['Date'].apply(lambda x: str(jdatetime.date.fromgregorian(date=x.date())))
    df_history = df_history.set_index('J-Date')
    # rearrange columns:
    df_history=df_history[['Date','Weekday','Y-Final','Open','High','Low','Close','Final','Volume','Value','No','Ticker','Name','Market']]
    cols = ['Y-Final','Open','High','Low','Close','Final','Volume','No','Value']
    df_history[cols] = df_history[cols].apply(pd.to_numeric, axis=1)
    #----------------------------------------------------------------------------------------------------------------------
    # Y-Final for new part of data could be 0 or 1000, we need to replace them with yesterday's final price:
    df_history['Final(+1)'] = df_history['Final'].shift(+1)          # final prices shifted forward by one day
    df_history['Market(+1)'] = df_history['Market'].shift(+1)        # market shifted forward by one day
    df_history['temp'] = df_history.apply(lambda x: x['Y-Final'] if((x['Y-Final']!=0)and(x['Y-Final']!=1000)) 
                                          else (x['Y-Final'] if((x['Market(+1)']==x['Market'])or(pd.isnull(x['Final(+1)']))) 
                                          else x['Final(+1)']),axis = 1)
    df_history['Y-Final'] = df_history['temp']
    df_history.drop(columns=['Final(+1)','temp','Market(+1)'],inplace=True)
    #-----------------------------------------------------------------------------------------------------------------------
    for col in cols:
        df_history[col] = df_history[col].apply(lambda x: int(x)) # convert to int because we do not have less than Rial
    #--------------------------------------------------------------------------------------------------------------------
    # Adjust price data:
    if(adjust_price):
        df_history['COEF'] = (df_history['Y-Final'].shift(-1)/df_history['Final']).fillna(1.0)
        df_history['ADJ-COEF']=df_history.iloc[::-1]['COEF'].cumprod().iloc[::-1]
        df_history['Adj Open'] = (df_history['Open']*df_history['ADJ-COEF']).apply(lambda x: int(x))
        df_history['Adj High'] = (df_history['High']*df_history['ADJ-COEF']).apply(lambda x: int(x))
        df_history['Adj Low'] = (df_history['Low']*df_history['ADJ-COEF']).apply(lambda x: int(x))
        df_history['Adj Close'] = (df_history['Close']*df_history['ADJ-COEF']).apply(lambda x: int(x))
        df_history['Adj Final'] = (df_history['Final']*df_history['ADJ-COEF']).apply(lambda x: int(x))
        df_history.drop(columns=['COEF','ADJ-COEF'],inplace=True)
    if(not show_weekday):
        df_history.drop(columns=['Weekday'],inplace=True)
    if(not double_date):
        df_history.drop(columns=['Date'],inplace=True)
    df_history.drop(columns=['Y-Final'],inplace=True)
    # slice requested time window:
    if(not ignore_date):
        df_history = df_history[start_date:end_date]
    return df_history


In [4]:
# Get tickers list from excel file
tickers = pd.read_excel('Stocks-List.xlsx')

In [5]:
# Get start and end date
start_date = '1400-01-01'
end_date = str(jdatetime.date.today())

In [6]:
# # Quick Check
# for ticker in tickers.iloc[600:]['Symbol']:
#     df = Get_Price_History(stock= ticker, start_date= start_date, end_date= end_date, double_date= True)
#     print(20*'-^-')
#     print(f'Symbol: {ticker}')
#     print(df.iloc[-1]['Close'])

In [7]:
# Price prediction part
import warnings
warnings.filterwarnings("ignore")

count = 1
sym_list = []
buy_list = pd.DataFrame()

for ticker in tickers['Symbol']:
    
    # Get historical data
    df = Get_Price_History(stock=ticker, start_date=start_date, end_date=end_date, double_date=True)
    
    # Set index
    df.set_index('Date', inplace=True)
    
    # Drop unnecessary columns
    df.drop(columns=['Open', 'High', 'Low', 'Final', 'Volume', 'Value', 'No', 'Ticker', 'Name', 'Market'], inplace=True)
    
    # Log Close Price
    df['Log_Close'] = np.log(df['Close'])
    
    # calculation of the log return
    df['Log_Ret'] = df['Log_Close'].diff()
    
    # Drop First NaN Value
    df.dropna(inplace= True)
    
    # Create model with Log Ret
    model = pm.auto_arima(df['Log_Ret'], trace=False, suppress_warnings=True, seasonal=False, error_action='ignore', stepwise=False)
    
    # Prediction
    n_pred = 3
    prediction = model.predict(n_periods=n_pred)
    
    # Needed to Compute Un-Differenced Predictions
    df['Shifted_Log_Close'] = df['Log_Close'].shift(1)
    
    # Create the future index
    pred_index_list = []
    pred_index =  df.index[-1] + pd.Timedelta(days=1)
    pred_index_list.append(pred_index)

    for i in range(n_pred-1):
        pred_index =  pred_index_list[-1] + pd.Timedelta(days=1)
        pred_index_list.append(pred_index)

    # Create pred dataframe
    pred = pd.DataFrame(index=pred_index_list)

    # Put prediction values into dataframe
    pred['Pred_with_diff'] = np.array(prediction)
    
    # Last Log_Close value
    last_Log_price = df.iloc[-1]['Log_Close']
    
    # Undo Differencing
    pred['Pred_Un_diff'] = pred['Pred_with_diff'] + last_Log_price
    
    # Undo Log
    pred['Pred'] = np.e ** (pred['Pred_Un_diff'])
    
    # Last Price
    Last_Price = df.iloc[-1]['Close']
    
    # First Prediction
    First_Pred = pred.iloc[-n_pred]['Pred']
    
    # Second Prediction
    Second_Pred = pred.iloc[-n_pred+1]['Pred']
    
    # Signal
    if First_Pred > Last_Price and Second_Pred > First_Pred:
        Signal = 'Strong Buy'
    elif First_Pred > Last_Price:
        Signal = 'Buy'
    else:
        Signal = 'Sell'
    
    # Print Signal
    if Signal == 'Strong Buy':
        sym_list.append(ticker)
        print(20*'-^-')
        print(f'[{count}] Symbol: {ticker}')
        print(f'Signal: {Signal}')
        print(f'Last Price: {Last_Price}')
        print(pred['Pred'].tail(n_pred))
    
    count += 1
    
buy_list['Symbol'] = sym_list
buy_list.to_excel('Buy_List.xlsx', header=False, index=False)

-^--^--^--^--^--^--^--^--^--^--^--^--^--^--^--^--^--^--^--^-
[1] Symbol: آ س پ
Signal: Strong Buy
Last Price: 11410.0
2023-10-08    11424.019692
2023-10-09    11436.658531
2023-10-10    11430.746164
Name: Pred, dtype: float64
-^--^--^--^--^--^--^--^--^--^--^--^--^--^--^--^--^--^--^--^-
[32] Symbol: بایکا
Signal: Strong Buy
Last Price: 26400.0
2023-10-05    26429.462057
2023-10-06    26469.293974
2023-10-07    26480.070792
Name: Pred, dtype: float64
-^--^--^--^--^--^--^--^--^--^--^--^--^--^--^--^--^--^--^--^-
[53] Symbol: بپیوند
Signal: Strong Buy
Last Price: 14760.0
2023-10-08    14781.706998
2023-10-09    14790.386834
2023-10-10    14816.430120
Name: Pred, dtype: float64
-^--^--^--^--^--^--^--^--^--^--^--^--^--^--^--^--^--^--^--^-
[69] Symbol: تفیرو
Signal: Strong Buy
Last Price: 42950.0
2023-08-30    43789.911085
2023-08-31    43805.983417
2023-09-01    43822.061648
Name: Pred, dtype: float64
-^--^--^--^--^--^--^--^--^--^--^--^--^--^--^--^--^--^--^--^-
[101] Symbol: ثشرق
Signal: Stro